# Soft-SPIN Healthcare — Step 1: importance → coupled indices (identification)

**Soft-SPIN variant (Contribution 2) applied to the healthcare case study (Contribution 3).**
Identification is *identical to regular SPIN* — **α does NOT appear in this step.** Soft-SPIN's only
change to SPIN (Algorithm 1, line 13) is *how* the coupled weights are suppressed at inference (`α·W`
instead of `0`), and that happens in Steps 2–4 (α grid `{0.0, 0.1, 0.3, 0.5, 0.7, 0.9, 1.0}`; α=0 = hard
SPIN, α=1 = baseline). The *set* of coupled weights is found here exactly as in the faithful healthcare
Step 1: importance `|W|·Σ|∇L|` (Michel et al. 2019) + the authors' **verbatim**
`SPIN.py::get_set_difference_mask` positional index-mask, `p=q=5e-7`, MLP target, grouped-layer scoring, fp16.

**Only the calibration data differs from the general-domain Soft-SPIN Step 1** — it is the three healthcare
sets built by the general pipeline (`cal_{fair,priv,general}_health.csv`; see `healthcare_case_study.md` §1):

- **D_f (fairness)** = EquityMedQA, 7 subsets, strict-unbiased `(question, answer)` targets, 128 stratified.
- **D_p (privacy / patient rights)** = MedSafetyBench category_4 `(harmful_request, safe_response)`, 128.
- **D_g (general medical)** = PubMedQA `pqa_labeled` `(question, long_answer)`, 128.

> **Sanity:** identification == regular SPIN, so `coupled_soft_health.json` should match the hard-SPIN
> operative set `coupled_indices_health.json` (**1163** weights) up to fp16 top-k ties (~0.3%; see
> `iter_health_results.md` §2.1). Step 4 below verifies this. If you want α=0 to be *bit-identical* to hard
> SPIN for the variant comparison, set `REUSE_HARD_COUPLED=True` (adopts the hard set verbatim, no GPU scoring).

**Inputs (add the general Step-1 output as a Kaggle Dataset input):** `cal_fair_health.csv`,
`cal_priv_health.csv`, `cal_general_health.csv` (and optionally `coupled_indices_health.json` for the sanity
check / reuse path). Internet ON (model). GPU **T4×2**. Runtime ~20 min.
**Output:** `coupled_soft_health.json` — feeds Soft-SPIN Step 2 (HSIC+ppl α-sweep) and Step 3 (generate).

## Step 0 — env + config

In [ ]:
import os
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF','expandable_segments:True')
import subprocess, sys, glob, gc, time, json
subprocess.run([sys.executable,'-m','pip','install','-q','transformers==4.46.3','datasets==2.18.0','sentencepiece'], check=False)
import random, numpy as np, pandas as pd, torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID='Qwen/Qwen2-7B-Instruct'      # healthcare case-study testbed (7B; SPIN mechanism is scale-dependent)
NSAMPLES=128
TARGET='mlp'
MAXLEN=1024          # safety cap (authors don't truncate; calibration samples are short)
LAYER_GROUP=4        # score 4 layers at a time (memory); scores identical to per-layer
P=5e-7; Q=5e-7       # Qwen2-7B extraction ratio (identical to the faithful/health Step 1)
SEED=0
DTYPE=torch.float16
ALPHAS=[0.0, 0.1, 0.3, 0.5, 0.7, 0.9, 1.0]   # recorded for provenance; α is APPLIED in Steps 2-4, not here
REUSE_HARD_COUPLED=False   # True -> adopt coupled_indices_health.json verbatim (α=0 == hard SPIN, no scoring)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
MODEL_TAG=MODEL_ID.split('/')[-1]
def find(name):
    for r in ['.','/kaggle/input','/kaggle/working']:
        h=glob.glob(os.path.join(r,'**',name),recursive=True)
        if h: return h[0]
    return None
def memline(tag=''):
    s=''
    for d in range(torch.cuda.device_count()):
        f,t=torch.cuda.mem_get_info(d); s+=f' GPU{d} {f/1e9:.1f}/{t/1e9:.1f}'
    print(f'  [{tag}]{s} GiB free')
print('Soft-SPIN Health Step 1 | model', MODEL_ID, '| p=q', P, '| reuse-hard', REUSE_HARD_COUPLED)

## Step 1 — locate healthcare calibration (built by the general Step 1)

Cast into the reproduction's `(prompt, response)` schema so the authors' loader semantics
(`prompt ⊕ response[1:]`, prompt masked to -100, loss on the response only) are reused unchanged.

In [ ]:
CF=find('cal_fair_health.csv'); CP=find('cal_priv_health.csv'); CG=find('cal_general_health.csv')
assert CF and CP and CG, 'add cal_{fair,priv,general}_health.csv (general Step-1 output) as a Kaggle input'
print('calibration CSVs:'); print(' ', CF); print(' ', CP); print(' ', CG)
HARD=find('coupled_indices_health.json')   # optional: hard-SPIN operative set, for sanity/reuse
print('hard-SPIN coupled set:', HARD if HARD else '(not provided — sanity check will be skipped)')

## Step 2 — authors' exact selection + scoring helpers (verbatim SPIN.py)

`get_set_difference_mask` / `coupled_indices_for_matrix` are copied verbatim from the reproduction; these are
the released code's positional index-mask, not the paper prose. α is absent — identification is regular SPIN.

In [ ]:
def target_names(m): return [n for n,p in m.named_parameters() if p.dim()==2 and 'weight' in n and TARGET in n]
def make_batches(path):
    d=pd.read_csv(path); out=[]
    for i in range(len(d)):
        ep=tokenizer(str(d['prompt'].iloc[i]),   return_tensors='pt').input_ids
        er=tokenizer(str(d['response'].iloc[i]), return_tensors='pt').input_ids[:,1:]   # authors' unconditional BOS drop
        inp=torch.cat([ep,er],1)[:,:MAXLEN].to(GEN_DEV); tar=inp.clone(); tar[:,:ep.shape[1]]=-100
        if (tar!=-100).sum()>0: out.append((inp,tar))
    return out

# ---- authors' EXACT SPIN.py logic (verbatim) ----
def get_set_difference_mask(p, q, W_metric1, W_metric2):
    top_p=int(p*W_metric1.shape[1]*W_metric1.shape[0])
    top_q=int(q*W_metric2.shape[1]*W_metric2.shape[0])
    top_p_indices=torch.topk(W_metric1.flatten(), top_p, largest=True)[1]
    top_q_indices=torch.topk(W_metric2.flatten(), top_q, largest=True)[1]
    unique_p=torch.unique(top_p_indices); unique_q=torch.unique(top_q_indices)
    mask_only_metric1=~torch.isin(unique_p, unique_q)
    mask_only_metric2=~torch.isin(unique_q, unique_p)
    mask_intersection=~(torch.ones_like(unique_p).bool() & mask_only_metric1 & mask_only_metric2)
    return mask_only_metric1, mask_only_metric2, mask_intersection, unique_q

def coupled_indices_for_matrix(W_general, W_fair, W_priv):
    _, m1, _, u1 = get_set_difference_mask(P, Q, W_general, W_fair)
    _, m2, _, u2 = get_set_difference_mask(P, Q, W_general, W_priv)
    return u1[m1 & m2]

@torch.enable_grad()
def score_group(gnames, batches):
    for n,p in model.named_parameters(): p.requires_grad_(n in gnames)
    acc={n:torch.zeros_like(model.get_parameter(n)) for n in gnames}
    for inp,tar in batches:
        model.zero_grad(set_to_none=True); model(input_ids=inp,labels=tar).loss.backward()
        for n in gnames:
            gr=model.get_parameter(n).grad
            if gr is not None: acc[n]+=gr.detach().abs()
    imp={n:(model.get_parameter(n).detach().abs()*acc[n]).cpu() for n in gnames}
    del acc; gc.collect(); torch.cuda.empty_cache()
    return imp
print('selection + scoring helpers ready.')

## Step 3 — identify coupled set → `coupled_soft_health.json` (checkpointed, resumable)

If `REUSE_HARD_COUPLED=True` and the hard-SPIN set is present, it is adopted verbatim (no model load, no
scoring — guarantees α=0 == hard SPIN exactly). Otherwise the model is loaded and the coupled set is
re-identified from the healthcare calibration (mirrors the general Soft-SPIN and Iter-SPIN Step 1).

In [ ]:
OUT='/kaggle/working/coupled_soft_health.json'
if REUSE_HARD_COUPLED and HARD:
    obj=json.load(open(HARD)); coupled=obj['coupled'] if 'coupled' in obj else obj
    coupled={k:[int(x) for x in v] for k,v in coupled.items()}
    meta={'model':MODEL_ID,'tag':MODEL_TAG,'domain':'healthcare',
          'variant':'soft-SPIN (identification = regular SPIN one-shot; adopted from hard-SPIN set)',
          'nsamples':NSAMPLES,'target':TARGET,'p':P,'q':Q,'alphas':ALPHAS,
          'coupled_count':sum(len(v) for v in coupled.values()),'coupled':coupled}
    json.dump(meta, open(OUT,'w'))
    print('REUSED hard-SPIN coupled set ->', OUT, '| weights', meta['coupled_count'])
else:
    tokenizer=AutoTokenizer.from_pretrained(MODEL_ID, use_fast=False, trust_remote_code=True)
    if tokenizer.pad_token is None: tokenizer.pad_token=tokenizer.eos_token
    model=AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=DTYPE, trust_remote_code=True,
            device_map='auto', max_memory={0:'14GiB',1:'14GiB','cpu':'28GiB'})
    model.eval()
    GEN_DEV=model.get_input_embeddings().weight.device
    print('model loaded | entry', GEN_DEV); memline('loaded')
    BATCHES={'f':make_batches(CF),'p':make_batches(CP),'g':make_batches(CG)}
    print('calibration batches | f',len(BATCHES['f']),'| p',len(BATCHES['p']),'| g',len(BATCHES['g']))

    names=target_names(model); nL=model.config.num_hidden_layers
    GROUPS=[list(range(g,min(g+LAYER_GROUP,nL))) for g in range(0,nL,LAYER_GROUP)]
    CK='/kaggle/working/coupled_soft_health_ckpt.pt'
    if os.path.exists(CK):
        st=torch.load(CK); coupled=st['coupled']; start=st['next_group']; print('resuming from group', start)
    else:
        coupled={}; start=0
    model.gradient_checkpointing_enable(); model.config.use_cache=False
    for gi in range(start, len(GROUPS)):
        grp=GROUPS[gi]; gnames=[n for n in names if any(f'.layers.{L}.' in n for L in grp)]
        t0=time.time()
        imp_f=score_group(gnames,BATCHES['f']); imp_p=score_group(gnames,BATCHES['p']); imp_g=score_group(gnames,BATCHES['g'])
        for n in gnames:
            fi=coupled_indices_for_matrix(imp_g[n], imp_f[n], imp_p[n])   # authors' exact selection
            if len(fi): coupled[n]=[int(x) for x in fi.tolist()]
        del imp_f,imp_p,imp_g; gc.collect(); torch.cuda.empty_cache()
        torch.save({'coupled':coupled,'next_group':gi+1}, CK)
        print(f'  group {gi+1}/{len(GROUPS)} done ({time.time()-t0:.0f}s) | coupled so far {sum(len(v) for v in coupled.values())}'); memline()
    model.gradient_checkpointing_disable()
    for p in model.parameters(): p.requires_grad_(False)
    meta={'model':MODEL_ID,'tag':MODEL_TAG,'domain':'healthcare',
          'variant':'soft-SPIN (identification = regular SPIN one-shot)',
          'nsamples':NSAMPLES,'target':TARGET,'p':P,'q':Q,'alphas':ALPHAS,
          'coupled_count':sum(len(v) for v in coupled.values()),'coupled':coupled}
    json.dump(meta, open(OUT,'w'))
    print('\nSTEP 1 DONE ->', OUT, '| coupled weights:', meta['coupled_count'])
print('This coupled set feeds Soft-SPIN Step 2 (HSIC+ppl α-sweep) and Step 3 (generate). ALPHA enters THERE, not here.')

## Step 4 — sanity check vs the hard-SPIN operative set (fp16 ties expected)

Identification == regular SPIN, so this soft set should match `coupled_indices_health.json` (1163) up to a
handful of fp16 top-k ties across two independent GPU runs (see `iter_health_results.md` §2.1). A large
mismatch (many matrices differing) would instead indicate a data/config drift — investigate.

In [ ]:
if HARD:
    hj=json.load(open(HARD)); hard=hj['coupled'] if 'coupled' in hj else hj
    hard={k:set(int(x) for x in v) for k,v in hard.items()}
    soft={k:set(int(x) for x in v) for k,v in json.load(open(OUT))['coupled'].items()}
    ht=sum(len(v) for v in hard.values()); st_=sum(len(v) for v in soft.values())
    hk,sk=set(hard),set(soft)
    diff=[(n,len(hard[n]),len(soft[n])) for n in (hk&sk) if hard[n]!=soft[n]]
    print(f'hard-SPIN total {ht} | soft total {st_} | net diff {st_-ht}')
    print(f'matrices only-in-hard {len(hk-sk)} | only-in-soft {len(sk-hk)} | differing (shared) {len(diff)} / {len(hk|sk)}')
    for n,lh,ls in diff: print(f'  {n}: hard={lh} soft={ls} | hard-only {len(hard[n]-soft[n])} soft-only {len(soft[n]-hard[n])}')
    ok = (not (hk ^ sk)) and len(diff)<=5
    print('VERDICT:', 'PASS (matches hard SPIN up to fp16 ties)' if ok else 'REVIEW (unexpected drift — check calibration/config)')
else:
    print('coupled_indices_health.json not provided — sanity check skipped. Add it as an input to verify n==hard SPIN.')